# LLM-driven self-play across scikit-decide domains

This tutorial drives several scikit-decide domains with `DSPyPolicy`, a
`DeterministicPolicySolver` whose policy is not computed offline but chosen
online, at every step, by prompting a real language model through
[DSPy](https://dspy.ai/). We use it here for real self-play: on a
single-agent domain the LLM plays the only agent step by step; on a
multi-agent domain (`RockPaperScissors`) the same solver instance plays
*every* agent, one real LLM call per agent per step, which is what makes it
"self-play" rather than just "an LLM policy".

**Prerequisite -- read this before running any code cell below.**
This notebook requires a real, already-running `TurboFieldfareServer`
(https://github.com/drumih/turbo-fieldfare) listening on
`http://127.0.0.1:8080` and serving the Gemma 4 model. Every cell below
makes a real HTTP call to that server -- there is no mock or stub LM here.
Start it beforehand, per `docs/OPENAI_SERVER.md` in the turbo-fieldfare
repository, e.g.:

```bash
cd ~/turbo-fieldfare
.build/release/TurboFieldfareServer --model scratch/gemma4.gturbo --port 8080 --max-context 4096
```

then confirm it is healthy from another terminal:

```bash
curl --silent --show-error http://127.0.0.1:8080/health
```

which should print `{"status":"ok"}`. If the server is not running, every
cell below that calls the LLM will fail with a connection error -- that is
expected, not a bug in this notebook.

## Setup

This notebook needs the scikit-decide `solvers` extra (for `dspy`). It is
not meant to run on Binder or Colab (it requires a local server process), so
we skip the usual Colab-install cell used by the other tutorials.

In [1]:
# dspy's lazy numpy loader recurses infinitely under numpy 2.x unless numpy
# is already a real, fully-loaded module in sys.modules before dspy is
# first imported -- see tests/conftest.py for the same guard applied there.
import numpy  # noqa: F401

import urllib.request

import dspy

from autofde_lab.hub.solver.dspy_policy.dspy_policy import DEFAULT_LM_API_BASE, DEFAULT_LM_MODEL
from autofde_lab.hub.solver.dspy_policy import DSPyPolicy
from autofde_lab.utils import rollout

We check the real server's health endpoint before doing anything else, so a
missing server fails fast with a clear message instead of a confusing
traceback deep inside `dspy`.

In [2]:
health_url = DEFAULT_LM_API_BASE.rsplit("/v1", 1)[0] + "/health"
with urllib.request.urlopen(health_url, timeout=5) as resp:
    assert resp.status == 200, f"TurboFieldfareServer not healthy at {health_url}"
    print(f"TurboFieldfareServer is healthy at {health_url}: {resp.read().decode()}")

TurboFieldfareServer is healthy at http://127.0.0.1:8080/health: {"status":"ok"}


In [3]:
# Real dspy.LM pointed at the real local TurboFieldfareServer, shared across
# every domain below (same DEFAULT_LM_MODEL used by
# tests/test_self_play_dspy_all_domains_chicago.py).
lm = dspy.LM(DEFAULT_LM_MODEL, api_base=DEFAULT_LM_API_BASE, api_key="local")
dspy.configure(lm=lm)
print(f"Configured dspy.LM: model={DEFAULT_LM_MODEL} api_base={DEFAULT_LM_API_BASE}")

Configured dspy.LM: model=openai/gemma-4-26b-a4b-it api_base=http://127.0.0.1:8080/v1


## Single-agent domains

`DSPyPolicy` declares a `MultiAgent` `T_domain`, but a `SingleAgent` domain
still autocasts onto it (`Solver.__init__`'s `autocast_all` presents the
single agent as a one-key dict `{"agent": ...}` under the hood). We drive
three real single-agent domains this way: `Maze`, `SimpleGridWorld`, and
`MasterMind` (the last one partially observable, so the LLM only ever sees a
`Score`, never the hidden state).

### Maze

In [4]:
from autofde_lab.hub.domain.maze import Maze


def maze_domain_factory() -> Maze:
    return Maze()


assert DSPyPolicy.check_domain(Maze())

with DSPyPolicy(domain_factory=maze_domain_factory, lm=lm) as maze_solver:
    maze_solver.solve()
    maze_rollout_domain = Maze()
    maze_legal_actions = maze_rollout_domain.get_action_space().get_elements()

    maze_episodes = rollout(
        maze_rollout_domain,
        solver=maze_solver,
        num_episodes=1,
        max_steps=3,
        render=False,
        verbose=False,
        return_episodes=True,
    )

observations, actions, values = maze_episodes[0]
print(f"Maze: {len(actions)} real LLM-chosen action(s) taken.")
for step, (action, value) in enumerate(zip(actions, values), start=1):
    assert action in maze_legal_actions
    print(f"  step {step}: action={action} value={value}")

2026-08-05 22:14:47,967 | autofde_lab.utils | INFO | The goal was not reached in episode 1.


Maze: 3 real LLM-chosen action(s) taken.
  step 1: action=Action.up value=Value(reward=-2, cost=2)
  step 2: action=Action.up value=Value(reward=-2, cost=2)
  step 3: action=Action.up value=Value(reward=-2, cost=2)


### SimpleGridWorld

In [5]:
from autofde_lab.hub.domain.simple_grid_world import SimpleGridWorld


def grid_domain_factory() -> SimpleGridWorld:
    return SimpleGridWorld()


assert DSPyPolicy.check_domain(SimpleGridWorld())

with DSPyPolicy(domain_factory=grid_domain_factory, lm=lm) as grid_solver:
    grid_solver.solve()
    grid_rollout_domain = SimpleGridWorld()
    grid_legal_actions = grid_rollout_domain.get_action_space().get_elements()

    grid_episodes = rollout(
        grid_rollout_domain,
        solver=grid_solver,
        num_episodes=1,
        max_steps=3,
        render=False,
        verbose=False,
        return_episodes=True,
    )

observations, actions, values = grid_episodes[0]
print(f"SimpleGridWorld: {len(actions)} real LLM-chosen action(s) taken.")
for step, (action, value) in enumerate(zip(actions, values), start=1):
    assert action in grid_legal_actions
    print(f"  step {step}: action={action} value={value}")

2026-08-05 22:14:48,001 | autofde_lab.utils | INFO | The goal was not reached in episode 1.


SimpleGridWorld: 3 real LLM-chosen action(s) taken.
  step 1: action=Action.up value=Value(reward=-2, cost=2)
  step 2: action=Action.up value=Value(reward=-2, cost=2)
  step 3: action=Action.up value=Value(reward=-2, cost=2)


### MasterMind

`MasterMind` is a `GoalPOMDPDomain`: the LLM's observation is a `Score`
(number of matching pins), never the hidden secret code, so this exercises
the partially-observable branch of `DSPyPolicy`.

In [6]:
from autofde_lab.hub.domain.mastermind import MasterMind


def mastermind_domain_factory() -> MasterMind:
    return MasterMind()


assert DSPyPolicy.check_domain(MasterMind())

with DSPyPolicy(domain_factory=mastermind_domain_factory, lm=lm) as mastermind_solver:
    mastermind_solver.solve()
    mastermind_rollout_domain = MasterMind()
    mastermind_legal_actions = mastermind_rollout_domain.get_action_space().get_elements()

    mastermind_episodes = rollout(
        mastermind_rollout_domain,
        solver=mastermind_solver,
        num_episodes=1,
        max_steps=3,
        render=False,
        verbose=False,
        return_episodes=True,
    )

observations, actions, values = mastermind_episodes[0]
print(f"MasterMind: {len(actions)} real LLM-chosen guess(es) taken (observation is a Score, not the hidden code).")
for step, (observation, action, value) in enumerate(zip(observations[1:], actions, values), start=1):
    assert action in mastermind_legal_actions
    print(f"  step {step}: guess={action} -> observation={observation} value={value}")

2026-08-05 22:14:48,016 | autofde_lab.utils | INFO | The goal was reached in episode 1.


MasterMind: 1 real LLM-chosen guess(es) taken (observation is a Score, not the hidden code).
  step 1: guess=(0, 0) -> observation=Score(total_bulls=2, total_cows=0) value=Value(reward=-1, cost=1)


## Multi-agent self-play: Rock-Paper-Scissors

This is the domain that gives the notebook its name: `RockPaperScissors` is a
real `MultiAgent` domain, and a single `DSPyPolicy` instance plays *both*
`player1` and `player2`, each with its own real LLM call at every step --
genuine LLM-vs-LLM self-play, not a single-agent policy relabelled.

In [7]:
from autofde_lab.hub.domain.rock_paper_scissors import RockPaperScissors

max_moves = 3


def rps_domain_factory() -> RockPaperScissors:
    return RockPaperScissors(max_moves=max_moves)


assert DSPyPolicy.check_domain(RockPaperScissors(max_moves=max_moves))

with DSPyPolicy(domain_factory=rps_domain_factory, lm=lm) as rps_solver:
    rps_solver.solve()
    rps_domain = rps_solver._domain
    rps_legal_action_spaces = rps_domain.get_action_space()

    rps_observation = rps_domain.reset()
    print(f"RockPaperScissors initial observation: {rps_observation}")
    for round_num in range(1, max_moves + 1):
        rps_action = rps_solver.sample_action(rps_observation)
        assert set(rps_action) == {"player1", "player2"}
        for agent, move in rps_action.items():
            assert move in rps_legal_action_spaces[agent].get_elements()
        rps_outcome = rps_domain.step(rps_action)
        rps_observation = rps_outcome.observation
        print(
            f"  round {round_num}: player1={rps_action['player1'].name} "
            f"player2={rps_action['player2'].name} "
            f"-> reward(player1)={rps_outcome.value['player1'].reward} "
            f"reward(player2)={rps_outcome.value['player2'].reward}"
        )
        if all(rps_outcome.termination.values()):
            print("  match finished.")
            break

Rollout domain not given. Using domain seen during solve instead.


Rollout domain not given. Using domain seen during solve instead.


Rollout domain not given. Using domain seen during solve instead.


RockPaperScissors initial observation: {'player1': <Move.rock: 0>, 'player2': <Move.rock: 0>}
  round 1: player1=paper player2=paper -> reward(player1)=0 reward(player2)=0
  round 2: player1=paper player2=scissors -> reward(player1)=-1 reward(player2)=1
  round 3: player1=paper player2=scissors -> reward(player1)=-1 reward(player2)=1
  match finished.


## Conclusion

`DSPyPolicy` lets a single, generic solver drive an unmodified
single-agent, partially-observable, or multi-agent scikit-decide domain
purely by prompting a real language model for each action, with no
domain-specific training and no offline `_solve` computation. On
`RockPaperScissors` the same solver instance plays every agent, which is what
turns "an LLM policy" into real LLM self-play. Every observation, action,
and reward printed above came from a real rollout against a real local
TurboFieldfareServer -- no mocked LM anywhere in this notebook.